# Question 1 -- Implementation checks (1.(e))

Only code lives here: an environment check, then one check per step of `FlexibleConsumerModel.build()` in `src/model.py`. Edit the `.py` file, save it, and re-run from the "build" cell downwards (`autoreload` picks up the changes). Figures, tables and commentary for 1.(f) are added once the model works.

In [21]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB

from src.data_loader import load_question
from src.model import FlexibleConsumerModel

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Environment check

In [22]:
print(f"Python {sys.version.split()[0]} | gurobipy {gp.gurobi.version()} | numpy {np.__version__} | pandas {pd.__version__}")
m0 = gp.Model("licence_check")
m0.Params.OutputFlag = 0
x0 = m0.addVar(ub=1)
m0.setObjective(x0, GRB.MAXIMIZE)
m0.optimize()
assert m0.Status == GRB.OPTIMAL and abs(x0.X - 1) < 1e-6
print("Gurobi licence OK")

Python 3.12.3 | gurobipy (13, 0, 3) | numpy 2.5.3 | pandas 3.0.5
Gurobi licence OK


## 1. Input data

In [23]:
data = load_question("Q1_caseA")
print(data.summary())
print("prices[:5]:", data.energy_price[:5], "| tariffs (imp, exp):", data.import_tariff, data.export_tariff)

Case                 : Q1_caseA
Consumer             : C_01_A
Energy price         : mean 1.32, min 0.85, max 2.50 DKK/kWh
Import/export tariff : 0.50 / 0.40 DKK/kWh
PV                   : 6.0 kW peak, 26.9 kWh available/day, marginal cost 1.41 DKK/kWh
Load bounds          : 0.0 - 6.0 kWh/h
Consumption utility  : 1.43
Min daily energy     : None
Reference profile    : no
Disutility lin/quad  : None / None
Battery              : no
Next-day forecasts   : yes
prices[:5]: [1.1  1.05 1.   0.9  0.85] | tariffs (imp, exp): 0.5 0.4


## 2. `build()`, step by step

**Legend: report (LaTeX) -> code.** In `build()`, `d = self.data`. Scalars have no index; arrays are indexed by hour `[t]`.

| Report | Code | Unit |
|---|---|---|
| $p_t$ | `d.energy_price[t]` | DKK/kWh |
| $\tau^{imp}$, $\tau^{exp}$ | `d.import_tariff`, `d.export_tariff` | DKK/kWh |
| $p^{imp}_t$, $p^{exp}_t$ | `p_imp[t]`, `p_exp[t]` (defined in `build()`) | DKK/kWh |
| $u^L$ | `d.consumption_utility` | DKK/kWh |
| $c^{PV}$ | `d.pv_marginal_cost` | DKK/kWh |
| $L^{\min}$, $L^{\max}$ | `d.load_min_kWh`, `d.load_max_kWh` | kWh/h |
| $PV^{\max}_t$ | `d.pv_available[t]` | kWh/h |
| $\ell_t$, $q^{PV}_t$ | `self.var["load"][t]`, `self.var["pv"][t]` | kWh/h |
| $q^{imp}_t$, $q^{exp}_t$ | `self.var["import"][t]`, `self.var["export"][t]` | kWh/h |
| $\lambda_t$, $\underline\mu^{L}_t$, $\overline\mu^{L}_t$, $\underline\mu^{PV}_t$, $\overline\mu^{PV}_t$, $\mu^{imp}_t$, $\mu^{exp}_t$ | columns `dual_balance`, `dual_load_lo`, `dual_load_up`, `dual_pv_lo`, `dual_pv_up`, `dual_imp_nonneg`, `dual_exp_nonneg` of `results.hourly` | DKK/kWh |

The dual column names assume the constraint families are called `balance`, `load_lo`, `load_up`, `pv_lo`, `pv_up`, `imp_nonneg`, `exp_nonneg` in `self.con`.

**Step 2 -- variables.** Four families of 24 free variables (`lb=-GRB.INFINITY`); every bound goes in a constraint (step 4). Step 1 (the effective prices `p_imp`, `p_exp`) is checked through the objective coefficients in step 3.

In [24]:
model = FlexibleConsumerModel(data).build()
for name in ["load", "pv", "import", "export"]:
    assert name in model.var, f"variable family '{name}' is missing"
    lbs = {v.LB for v in model.var[name].values()}
    print(f"{name:7s} n={len(model.var[name])}  lb={lbs}")
    assert len(model.var[name]) == 24 and lbs == {-GRB.INFINITY}, f"'{name}': expected 24 variables with lb=-GRB.INFINITY"

NameError: name 'u' is not defined

**Step 3 -- objective.** Maximisation; the check compares the coefficient of each variable in hour 0 with the expected one.

In [ ]:
assert model.m.ModelSense == GRB.MAXIMIZE, "the objective must be a maximisation"
p_imp0 = data.energy_price[0] + data.import_tariff
p_exp0 = data.energy_price[0] - data.export_tariff
expected = {"load": data.consumption_utility, "import": -p_imp0, "export": p_exp0, "pv": -data.pv_marginal_cost}
for name, exp in expected.items():
    got = model.var[name][0].Obj
    print(f"{name:7s} coefficient {got:8.3f}   expected {exp:8.3f}")
    assert abs(got - exp) < 1e-9, f"wrong objective coefficient for '{name}'"

**Step 4 -- constraints.** Seven families of 24 constraints: balance, load bounds (2), PV limits (2), non-negativity of import and export.

In [ ]:
for name, c in model.con.items():
    print(f"{name:12s} n={len(c)}  sense={ {con.Sense for con in c.values()} }")
print("total constraints:", model.m.NumConstrs)
assert len(model.con) == 7 and model.m.NumConstrs == 168, "expected 7 families x 24 hours = 168 constraints"

**Step 5 -- solve.**

In [ ]:
results = model.solve()
print(results)
hr = results.hourly
display(hr.round(3))

**Step 6 -- primal checks.**

In [ ]:
p_imp = data.energy_price + data.import_tariff     # effective import price, DKK/kWh
p_exp = data.energy_price - data.export_tariff     # effective export price, DKK/kWh

# TODO (a): power balance. Print the maximum absolute value of  load - (pv + import - export).
# TODO (b): bounds. Check that Lmin <= load <= Lmax and 0 <= pv <= pv_available in every hour.
# TODO (c): no simultaneous import and export. Count the hours with both > 1e-6 (expect 0).
# TODO (d): objective. Compute utility = sum(uL * load) and
#           procurement cost = sum(p_imp*import - p_exp*export + cPV*pv);
#           check that utility - cost equals results.objective.

**Step 7 -- dual checks.**

In [ ]:
# Dual columns are named "dual_<name>", with <name> the key used in self.con[...] in build().
print([c for c in hr.columns if c.startswith("dual_")])

# TODO (a): in import hours (import > 1e-6) the dual of the balance should equal p_imp. Print max |dual - p_imp|.
# TODO (b): same for export hours: the dual of the balance should equal p_exp.
# TODO (c): check the four stationarity conditions of your 1.(b), one at a time, e.g. for the load:
#           -uL + lambda - mu_L_lo + mu_L_up == 0   (with your sign convention). Print each maximum error.
# TODO (d): print the sign of every dual (all mu should be >= 0). If not, revisit the sign of the
#           inequalities in build() (write them as  expression <= 0).

**Step 8 -- reproducibility.** The same case must run from the terminal with one command.

In [ ]:
import subprocess
proc = subprocess.run([sys.executable, str(ROOT_DIR / "main.py"), "--question", "Q1_caseA"],
                      cwd=ROOT_DIR, capture_output=True, text=True)
print(proc.stdout[-700:])
assert proc.returncode == 0, proc.stderr